# Quantra — Barbershop Fast Loop (Colab)

**Rulebook (for the Risk Doctor / any LLM):** `docs/MLP_INTERPRETABILITY_LAYER.md`.
**Master manual:** `docs/PROJECT_GUIDE.md` — §4.10 (this loop), §4.11 (Policy Registry),
§4.12 (Runtime Override System), §4.13 (Repo Safety), §4.14 (Colab GPU Setup).

**What this is.** The *Barbershop* — *"get the haircut before going to school."* You pick a
small window of FTMO challenge days, run the policy fast, watch what breaks (which days fail
and why), make **one** educated edit via the `OVERRIDES` dict, and repeat. The sole mission of
the policy is to **pass FTMO-style challenges**: hit **+2.5%/day** without breaching a
**−4% trailing wall** on real MT5 bars.

**Two modes, entered freely (not sequential):**
- **Barbershop mode** (this notebook) — fast, operator-driven diagnose-and-shape.
- **Full Training mode** (`colab/Quantra_Train.ipynb`) — long walk-forward runs that
  *generalize* the shaped policy across regimes.

**Repo safety.**
- **ACTIVE WORK** (edits/commits here): `github.com/monty313/RL-model-trading-bot-ppo-mlp_Claude-`
- **FALLBACK / SAFE RESTORE** (never edited — emergency revert only): `github.com/monty313/final-rl-model-6_13`

**Known gaps (always honest):** (1) **no real *trained* model yet** — the #1 active item. Cell 5 now
runs the REAL env+policy (C21), so the metrics/Leaderboard are real, but the policy is **untrained**
until M8 wires the PPO update + checkpoint I/O, so it won't PASS yet; (2) the sim models ONE wall,
real FTMO has TWO (the −10% max is an observation via C12, not enforced); (3) Screen 1 is a labelled
demo curve until a real pass-rate series is logged; (4) trade-autopsy attribution is input×gradient,
not SHAP.

> ⚠️ **Mostly real, two stubs left.** The loop now produces REAL per-day metrics — Cell 5 calls
> `barbershop_runner.run_pass()` against a `TradingEnv` (C21), and Cell 6 writes the real Policy
> Registry + Leaderboard. The remaining `TODO(wire)` stubs are **checkpoint I/O** (`agent.load`/
> `agent.checkpoint`) and the **PPO weight update** (M8) — until those land the policy is untrained.

## Cell 1 — Mount Drive · clone the ACTIVE repo · install deps · race the hardware

In [ ]:
# ── Cell 1 ─ Startup (run once per Colab session; guarded so re-runs are instant) ──
# FTMO framing: every second of CPU setup is a second not spent learning to pass the
# challenge, so the slow steps here are guarded to run ONCE per session.
import os, sys, subprocess, time

REPO_DIR = "quantra_repo"
# ACTIVE WORK repo (we edit/commit here). FALLBACK = final-rl-model-6_13 (NEVER edited) —
# see PROJECT_GUIDE.md §4.13 Repo Safety Protocol.
ACTIVE_REPO = "https://github.com/monty313/RL-model-trading-bot-ppo-mlp_Claude-.git"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", ACTIVE_REPO, REPO_DIR], check=True)
if os.path.basename(os.getcwd()) != REPO_DIR:
    os.chdir(REPO_DIR)

# Mount Drive once (price data + checkpoints persist across Colab disconnects).
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive mount skipped (not on Colab?):", e)

# Install only the extras Colab doesn't ship. Sentinel guard → instant on re-run.
_SENTINEL = "/content/.quantra_deps_installed"
if not os.path.exists(_SENTINEL):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "gdown", "pyarrow", "psutil", "optuna", "seaborn",
                    "statsmodels", "gymnasium", "tqdm", "pyngrok"], check=True)
    import torch
    if torch.cuda.is_available():
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nvidia-ml-py"], check=True)
    open(_SENTINEL, "w").close()

# §4.14 PERFORMANCE RULE: call the hardware auto-optimizer at STARTUP so training uses
# ~80% of whichever device is genuinely faster → more passes validated per GPU-hour.
from quantra.runtime import RuntimeConfig, plan, print_report, UtilizationMonitor, ensure_dirs
ensure_dirs()
RT = RuntimeConfig()
with UtilizationMonitor(interval=0.25) as _mon:
    HW = plan(state_dim=RT.nominal_state_dim, hw=RT.hardware)
print_report(HW)
print("Device:", HW.device_label, "| n_envs:", HW.n_envs)

## Cell 2 — Build the data cache ONCE (CSV → features → memmap), reuse forever

In [ ]:
# ── Cell 2 ─ Cache-once data pipeline ──
# FTMO framing: the policy can only learn to pass on bars it can see FAST. Parse +
# feature-build is the slow CPU step; do it once, cache it, keep every pass GPU-bound.
from quantra.market_pipeline.data_loader import load_symbol
from quantra.env.trading_env import prepare_symbol_data

SYMBOL = "EURUSD"   # the binding case for real-bar training (#1 gap — see PROJECT_GUIDE §7)

# path=None → loader resolves bars itself: Parquet cache → Drive mount → gdown by ID.
df, rep = load_symbol(SYMBOL, path=None)
print(f"[load] {len(df):,} bars  {df.index.min()} -> {df.index.max()}  source={rep.source}")

# prepare_symbol_data builds the locked 207-feature observation (memmap-cached by the
# feature builder). Re-running is cheap once the cache exists.
SD = prepare_symbol_data(df, symbol=SYMBOL)
print(f"[features] T={len(SD.close):,}  warmup={SD.valid_from:,}  usable={len(SD.close)-SD.valid_from:,}")

## Cell 3 — OPERATOR EDITS HERE: the window + the OVERRIDES

In [ ]:
# ── Cell 3 ─ INPUTS + OVERRIDES (the only cell you normally edit) ──

# ---- INPUTS ------------------------------------------------------------------
POLICY_NAME         = "v1-baseline"    # operator label; the FINAL saved name is AUTO-GENERATED
                                       # from the OVERRIDES diff (Cell 6), never hand-typed.
START_DATE          = "2023-03-01"     # where in the data to start the challenge window
N_DAYS              = 8                 # C10 episode length: trading days on ONE continuous account
                                       # (= EVAL_DAYS; the account carries forward across these days)
N_PASSES            = 20                # how many times the loop repeats over those N_DAYS
CHECKPOINT_INTERVAL = 5                 # save weights + telemetry + registry every N passes
RESUME_FROM         = None             # path to a checkpoint to continue from, or None = fresh

# ---- OVERRIDES (injected at launch — NO edits to config.py / laws.py needed) --
# Every knob is a runtime override saved to the Policy Registry, so you always know which
# configuration produced which result. Keys here MUST match registry._NAME_ORDER/_SHORT (the
# challenge/reward field names + 'training_phase' as the 'free'/'constrained' string). Anything you
# leave at its baseline default produces NO name token (Cell 6 diffs vs the live config defaults).
# Comment each change relative to PASSING the challenge.
OVERRIDES = {
    # Enforcement phase for the 3 market-condition signals (volatility / spread / stationarity).
    # "free" (DEFAULT) = OBSERVATION-ONLY: the bot SEES the signals and LEARNS to trade both
    # stationary AND non-stationary conditions itself — no hard blocking. This is what lets the
    # policy trade enough to learn to pass (the old hard gates shut ~98.7% of opens on EURUSD).
    # "constrained" = the stationarity signal re-enforces (blocks opens when non-stationary) —
    # a LATE hardening step only. (engine.py reads config.TRAINING_PHASE.)
    "training_phase": "free",

    # Training wheels (operator counter-trend OPEN blocks on 30m+4H).
    "training_wheels": True,            # True = enforce, False = remove

    # Challenge numbers — passed to make_challenge(); YOU set these.
    "daily_target_pct": 2.5,            # daily profit goal (% of that day's OPENING balance)
    "daily_risk_pct":   4.0,            # the trailing-wall risk allowed to make it
    "permanent_dd_pct": 10.0,           # the -10% permanent max-overall-loss wall — OBSERVATION
                                        # ONLY (C12 dist_to_perm_dd scalar), NOT enforced in training.

    # C10/C11 — multi-day episode + the failed-day penalty.
    "failed_day_penalty": 5.0,          # C11 weight: LARGE end-of-day reward hit, proportional to how
                                        # far below the daily target the day finished (worst for
                                        # wall-hit/locked-out days). Big by design — drives CONSISTENCY.

    # Reward shaping weights (C16/C17) — operator-tunable NOW via RewardConfig. UNCOMMENT to shape
    # HOW the bot passes. These change a multiplier / what a layer computes, NOT the reward LAYER
    # arrangement, so they keep Layer-0 dominance AND the compatibility signature (resuming an old
    # policy across a weight/math change is safe — see the COMPATIBILITY MAP in README §4).
    # "daily_progress_weight": 1e-3,    # the consistency driver (default 1e-3; raise to push harder)
    # "trade_quality_weight":  5e-5,    # reward banking winners / penalize round-tripping a winner
    # "step_pnl_weight":       1e-4,    # small per-bar in-position bonus
    # "drawdown_pain_weight":  5e-4,    # pain-zone penalty approaching the wall

    # ── NOTE (2026-06-18/19 redesign) ────────────────────────────────────────
    # The 3 former "gates" became OBSERVATION signals: the old gate-threshold knobs
    # (adf_p_value_threshold / atr_min_multiplier / spread_max_pips) were REMOVED. A breached day
    # no longer ENDS the episode — it locks out the rest of that day and resets at midnight (C10);
    # the account carries forward. Operator-tunable REWARD weights ARE now wired (C16 RewardConfig +
    # C17 re-pointed math) — add any of the weight keys above to shape the policy.
}
print("INPUTS + OVERRIDES set. Now run Cells 4-8.")

## Cell 4 — Load or init the policy + checkpoint compatibility check

In [ ]:
# ── Cell 4 ─ Policy init + compatibility gate (REAL Policy Registry) ──
import json, datetime
from pathlib import Path
from quantra.learning_system.ppo_agent.agent import PPOAgent
from quantra.learning_system.policy_registry import (
    compatibility_signature, default_reward_layer_keys, default_law_fingerprint,
    check_compatibility, CompatibilityError)

REGISTRY_DIR = Path("artifacts/policy_registry")
STATE_DIM = RT.nominal_state_dim   # 🔴 LOCKED observation width (207, or 189 w/o raw inputs)

# The compatibility signature is computed by the REAL registry from the LIVE owner files — see the
# 📍 COMPATIBILITY MAP in registry.py / README §4: STATE_DIM (schema.py) + the reward decompose
# L-keys (reward.py) + laws.LAW_NAMES. Operator-tunable weights (C16) and re-pointed term math (C17)
# keep it STABLE (same dim + L-keys + laws), so old policies stay resume-safe; only changing the dim,
# the reward LAYER arrangement, or the law set forces a fresh start. The helpers read code live, so
# this can never silently drift from what the policy was actually trained on.
CUR_SIG = compatibility_signature(STATE_DIM, default_reward_layer_keys(), default_law_fingerprint())

agent = PPOAgent()   # reads STATE_DIM dynamically — no hardcoded input dim
BASE_POLICY = None   # the resumed policy's name (drives the v<N> bump in the auto-name) when resuming

if RESUME_FROM:
    man_path = Path(RESUME_FROM).with_name("manifest.json")
    if man_path.exists():
        old = json.loads(man_path.read_text())
        BASE_POLICY = old.get("policy_name")
        # Raises CompatibilityError (old checkpoint is kept, NEVER overwritten) on any mismatch.
        check_compatibility(old.get("compatibility_signature", ""), CUR_SIG,
                            detail="STATE_DIM / reward-layer arrangement / law set changed")
    # TODO(wire): agent.load(RESUME_FROM) once fast-loop checkpoint I/O is connected.
    print("Resuming from", RESUME_FROM, "(compatibility OK); base policy:", BASE_POLICY)
else:
    print("Starting a FRESH policy (RESUME_FROM is None).")
print("Compatibility signature:", CUR_SIG)

## Cell 5 — THE BARBERSHOP LOOP (live output · clean stop-on-interrupt)

In [ ]:
# ── Cell 5 ─ THE BARBERSHOP LOOP (REAL metrics from TradingEnv · clean stop-on-interrupt) ──
# Hit Colab's STOP any time — the interrupt is caught and a clean checkpoint is saved before exit.
# Per pass, run_pass() drives the DETERMINISTIC policy over N_DAYS on ONE continuous account (C10)
# and returns one REAL scoreboard row per day. COUPLING -> quantra/learning_system/barbershop_runner.py
# (run_pass / slice_symbol_data) -> quantra/env/trading_env.py (TradingEnv) + ppo_agent/agent.py.
import numpy as np
from quantra.learning_system.barbershop_runner import run_pass, slice_symbol_data

# Window the cached features at START_DATE WITHOUT a re-build (SD rows align 1:1 with df). Each pass
# gets a FRESH env (the account resets per pass); the same feature window is reused across passes.
_start_i = int(df.index.searchsorted(np.datetime64(START_DATE))) if START_DATE else 0
DATA = {SYMBOL: slice_symbol_data(SD, _start_i)}
print(f"Window: from {START_DATE} (bar {_start_i}) | {len(DATA[SYMBOL].close):,} bars | episode={N_DAYS} days\n")

def print_pass_table(p, n_passes, day_results):
    print(f"Pass {p}/{n_passes}")
    for r in day_results:
        verdict = "PASS  " if r["passed"] else "FAIL  "
        dd = f"DD {r['dd_pct']:.1f}%" + (" BREACHED" if r["breached"] else "")
        print(f"  Day {r['day']}: {verdict}| {r['pnl_pct']:+.1f}% | {dd} | {r['trades']} trades")
    npass = sum(r["passed"] for r in day_results)
    avg = lambda k: sum(r[k] for r in day_results) / len(day_results)
    print(f"  Summary: {npass}/{len(day_results)} passed | Avg P&L: {avg('pnl_pct'):+.1f}% | "
          f"Avg DD: {avg('dd_pct'):.1f}% | Avg trades/day: {avg('trades'):.1f}")
    # In PHASE_FREE the bot is never blocked — a LOW trade count means the POLICY chose not to trade
    # (shape it via reward), NOT a gate lockout. gate_block_rate is >0 only in PHASE_CONSTRAINED.

def summarize_pass(p, day_results):
    avg = lambda k: round(sum(r[k] for r in day_results) / len(day_results), 3)
    return {"pass": p, "days_passed": sum(r["passed"] for r in day_results),
            "days_failed": sum(not r["passed"] for r in day_results),
            "avg_pnl": avg("pnl_pct"), "avg_dd": avg("dd_pct"),
            "breach_count": sum(r["breached"] for r in day_results),
            "avg_gate_block_rate": avg("gate_block_rate")}   # 0.0 in PHASE_FREE

PASS_HISTORY = []   # consumed by Cell 6 → performance.json
print("NOTE: the policy is UNTRAINED until M8 wires training — these metrics are REAL (what this")
print("policy actually did on the env), not placeholders; they just won't PASS until training lands.\n")
try:
    for p in range(1, N_PASSES + 1):
        day_results = run_pass(agent, DATA, OVERRIDES, N_DAYS, deterministic=True)
        if not day_results:
            print(f"Pass {p}: no days produced — check START_DATE / N_DAYS vs the data length."); break
        print_pass_table(p, N_PASSES, day_results)
        PASS_HISTORY.append(summarize_pass(p, day_results))
        if p % CHECKPOINT_INTERVAL == 0:
            print(f"  [checkpoint] pass {p}: persisting weights + telemetry + registry…")
            # TODO(wire): agent.checkpoint(<drive path>) so a Colab disconnect resumes here.
except KeyboardInterrupt:
    print("\n[stop] interrupt caught — saving a clean checkpoint before exit. Weights are safe.")
    # TODO(wire): agent.checkpoint(<drive path>) + re-run Cell 6 so nothing is lost.
print("\nLoop finished. Run Cell 6 to persist telemetry + the Policy Registry entry.")

## Cell 6 — Auto-emit telemetry + write the Policy Registry entry (AUTO-NAMED)

In [ ]:
# ── Cell 6 ─ Persist telemetry + the Policy Registry entry (REAL registry module) ──
# The policy name is DERIVED from the OVERRIDES diff vs the live baseline (config dataclass defaults)
# by registry.build_card() — NEVER hand-typed. record_pass() fills performance.json; save() writes the
# 3 files (manifest/performance/compatibility); the Leaderboard ranks every saved policy by how well
# it PASSES. COUPLING -> quantra/learning_system/policy_registry/registry.py (the contract is
# artifacts/policy_registry/README.md); OVERRIDES comes from Cell 3, BASE_POLICY/CUR_SIG from Cell 4.
from quantra.learning_system.policy_registry import build_card, PassRecord, Leaderboard

card = build_card(overrides=OVERRIDES, state_dim=STATE_DIM,
                  data_window={"start": START_DATE, "n_days": N_DAYS}, base_policy=BASE_POLICY)
for s in PASS_HISTORY:                       # PASS_HISTORY rows already match PassRecord's schema
    card.record_pass(PassRecord.from_dict(s))
assert card.compatibility_signature == CUR_SIG   # what we SAVE == Cell 4's resume gate (no drift)
pdir = card.save()                           # -> artifacts/policy_registry/<auto-name>/{manifest,performance,compatibility}

print("Auto-generated policy name:", card.policy_name, " (operator label was:", POLICY_NAME, ")")
print(f"  passes={card.n_passes_completed}  overall_pass_rate={card.overall_pass_rate():.3f}  best_pass={card.best_pass()}")
print("Registry written  ->", pdir)

# Telemetry for the Barbershop dashboard. Canonical real-bar PRODUCER: scripts/emit_real_telemetry.py.
RUN_ID = card.policy_name + "_" + datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
tel_dir = Path("artifacts/telemetry"); tel_dir.mkdir(parents=True, exist_ok=True)
with (tel_dir / f"{RUN_ID}.jsonl").open("w") as f:
    for s in PASS_HISTORY:
        f.write(json.dumps(s) + "\n")
print("Telemetry written ->", tel_dir / f"{RUN_ID}.jsonl")

# ── The Leaderboard: every saved policy ranked by passing quality ──
# (pass-rate -> best days-passed -> fewest breaches -> most recent). This is how you pick which
# policy to RESUME or PROMOTE to Full Training. Reads every <name>/performance.json under the registry.
print("\n" + Leaderboard.from_dir().render())

## Cell 7 — Inline Barbershop charts (Screen 1 wall · Screen 3 day replay)

In [ ]:
# ── Cell 7 ─ Inline charts via barbershop/figures.py (the full 5 screens are in Cell 8) ──
from barbershop import figures, data as bdata

# Auto-source: REAL telemetry if a run exists under artifacts/telemetry/, else mock.
bundle = bdata.load_bundle(source=bdata.available_source())
print("Barbershop data source:", bundle.get("source", "?"))

# Screen 1 — Training Wall (honest 'demo curve' label until a real pass-rate series logs).
wall = bundle.get("training_wall", {}) or {}
figures.training_wall_figure(wall.get("iterations", []), wall.get("pass_rate", [])).show()

# Screen 3 — Day Replay candlesticks for the worst day. Exact bundle keys are defined in
# barbershop/contract.py + barbershop/data.py; we read defensively so the skeleton degrades
# gracefully if a key name differs.
days = bundle.get("days") or bundle.get("scoreboard") or []
if days:
    d0 = days[0]
    prices, trades = d0.get("prices"), d0.get("trades", [])
    if prices is not None:
        figures.candlestick_figure(prices, trades, tf="1m").show()
print("Screen 2 scoreboard (HTML cards) + Screens 4-5 + Risk Doctor → open the dashboard (Cell 8).")

## Cell 8 — Open the FULL Barbershop dashboard via an ngrok tunnel (READ-ONLY)

In [ ]:
# ── Cell 8 ─ ngrok tunnel to the Dash app (all 5 screens + Risk Doctor) ──
# The Barbershop is READ-ONLY: it NEVER changes training, rewards, the policy, or execution.
import threading
from pyngrok import ngrok
from barbershop.dashboard import make_app
from barbershop.data import available_source

PORT = 8050
# Set your token once (https://dashboard.ngrok.com): export NGROK_AUTHTOKEN=... before Colab,
# or paste it here. Without a token ngrok allows a limited free tunnel.
NGROK_AUTHTOKEN = os.environ.get("NGROK_AUTHTOKEN", "")
if NGROK_AUTHTOKEN:
    ngrok.set_auth_token(NGROK_AUTHTOKEN)

def _serve():
    make_app(source=available_source()).run(host="0.0.0.0", port=PORT, debug=False)

threading.Thread(target=_serve, daemon=True).start()
time.sleep(3)
public_url = ngrok.connect(PORT, "http").public_url
print("Barbershop dashboard (all 5 screens + Risk Doctor):", public_url)